# Analyse reproductible du benchmark - Togo AI Benchmark

Ce notebook charge les évaluations automatiques et les annotations humaines, calcule des agrégations par modèle et par question, et produit des visualisations et exports réutilisables. Exécutez les cellules dans l'ordre.

In [ ]:
# Imports
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

print('Libraries loaded')

In [ ]:
# Helper: read JSONL files
def read_jsonl(path):
    items = []
    p = Path(path)
    if not p.exists():
        return items
    with p.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except Exception:
                # skip malformed line
                continue
    return items

print('Helper ready')

In [ ]:
# Load automatic evaluations from outputs/evaluations
eval_dir = Path('outputs/evaluations')
eval_files = sorted(eval_dir.glob('*.jsonl'))
auto_records = []
for f in eval_files:
    auto_records += read_jsonl(f)
print(f'Loaded {len(auto_records)} automatic evaluation records from {len(eval_files)} files')

In [ ]:
# Normalize automatic evaluations into a DataFrame: one row per (question, model, metric)
rows = []
for r in auto_records:
    qid = (r.get('question_metadata') or {}).get('question_id') or r.get('question_id')
    # prefer evaluated_by.model then generated_by.model
    model = (r.get('evaluation_metadata') or {}).get('evaluated_by', {}).get('model') or (r.get('evaluation_metadata') or {}).get('generated_by', {}).get('model')
    scores = r.get('evaluation_scores') or {}
    for metric, obj in scores.items():
        score = obj.get('score') if isinstance(obj, dict) else obj
        try:
            score = float(score)
        except Exception:
            continue
        rows.append({'question_id': qid, 'model': model or 'unknown', 'source': 'auto', 'metric': metric, 'score': score})
auto_df = pd.DataFrame(rows)
auto_df.head()

In [ ]:
# Load human annotations
human_dir = Path('outputs/human_annotations')
human_files = sorted(human_dir.glob('*.jsonl'))
hum_rows = []
for f in human_files:
    for r in read_jsonl(f):
        qid = r.get('question_id')
        # model mapping is optional here; if you have a raw_map, merge later
        model = r.get('model') or 'unknown'
        scores = r.get('scores') or {}
        for metric, val in scores.items():
            try:
                s = float(val)
            except Exception:
                continue
            hum_rows.append({'question_id': qid, 'model': model, 'source': 'human', 'metric': metric, 'score': s})
human_df = pd.DataFrame(hum_rows)
print(f'Loaded {len(human_df)} human annotation rows from {len(human_files)} files')
human_df.head()

In [ ]:
# Combine and summarize
combined = pd.concat([auto_df, human_df], ignore_index=True, sort=False) if not human_df.empty else auto_df.copy()
summary = combined.groupby(['source','model','metric']).score.agg(['mean','count']).reset_index()
summary_pivot = summary.pivot_table(index=['model','metric'], columns='source', values='mean')
summary_pivot.head()

In [ ]:
# Visual example: compare auto vs human for C1_exactitude_source_verite if present
metric = 'C1_exactitude_source_verite'
if metric in combined['metric'].unique():
    dfm = combined[combined.metric==metric].groupby(['source','model']).score.mean().reset_index()
    plt.figure(figsize=(10,5))
    sns.barplot(data=dfm, x='model', y='score', hue='source')
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Comparaison des moyennes pour {metric}')
    plt.tight_layout()
    plt.show()
else:
    print(f'Metric {metric} not found in data')

In [ ]:
# Export summary CSV
out_dir = Path('outputs/reports')
out_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(out_dir / 'per_model_metric_summary.csv', index=False)
print('Exported per_model_metric_summary.csv')

## Remarques finales
- Pour une attribution précise des annotateurs à un `model`, créez un `raw_map` (response_id → generating model) en chargeant `outputs/raw/*.jsonl` et joignez les `response_id` des annotations humaines.
- Recommander d'exécuter ce notebook sur un échantillon pilote (100 items × 3 annotateurs) avant un run complet.